# OSM Notes Classifier: export models for CPU

Turns the trained models (category and theme) into **ONNX int8** files so the
server can classify new notes on CPU, without a GPU and without PyTorch.

- int8 makes each model ~4x smaller (~2.2 GB → ~0.6 GB) and 2–3x faster on CPU.

In [1]:
!pip install -q "transformers>=4.46,<5" sentencepiece protobuf onnx onnxscript onnxruntime

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 1. Settings

In [2]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/osm-notes-training"
RUN_NAME = "ft_v2_context"
MODEL = "xlm-roberta-large"     # best model of the run
TARGETS = ["category", "theme"]
WORK = "/content/onnx"           # local disk; copied to Drive at the end
OUT = f"{DRIVE}/{RUN_NAME}/onnx"
MAX_LEN = 128                    # same as training
THREADS = 4                      # CPU threads of the server, for the speed test

Mounted at /content/drive


## 2. Export to ONNX and quantize to int8

`reduce_range` and per-channel weights keep int8 accurate on CPUs without VNNI
(the server is an Intel Skylake).

In [3]:
import json
import shutil

import torch
from onnxruntime.quantization import QuantType, quantize_dynamic
from transformers import AutoModelForSequenceClassification, AutoTokenizer


class Logits(torch.nn.Module):
    """Return only the logits, so the ONNX graph has one plain output."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits


def export(target):
    src = f"{DRIVE}/{RUN_NAME}/{target}/{MODEL}/model"
    fp32_dir, dst = f"{WORK}/fp32/{target}", f"{WORK}/{target}"
    os.makedirs(fp32_dir, exist_ok=True)
    os.makedirs(dst, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(src)
    model = AutoModelForSequenceClassification.from_pretrained(src).eval()
    enc = tok(["Missing bakery on this corner"], return_tensors="pt")
    args = (enc["input_ids"], enc["attention_mask"])
    fp32 = f"{fp32_dir}/model.onnx"
    common = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
                  dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                                "attention_mask": {0: "batch", 1: "seq"},
                                "logits": {0: "batch"}})
    try:
        torch.onnx.export(Logits(model), args, fp32, opset_version=17, dynamo=False, **common)
    except Exception as e:   # newer PyTorch without the old exporter
        print("Old exporter failed, using dynamo:", type(e).__name__, e)
        torch.onnx.export(Logits(model), args, fp32, opset_version=18, dynamo=True,
                          external_data=True, **common)

    quantize_dynamic(fp32, f"{dst}/model.onnx", weight_type=QuantType.QInt8,
                     per_channel=True, reduce_range=True)

    tok.save_pretrained(dst)   # tokenizer.json is what the server reads
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    meta = {"run_name": RUN_NAME, "model": MODEL, "target": target, "labels": labels,
            "max_len": MAX_LEN, "input": "[template] [poi_type] [context] user_text"}
    json.dump(meta, open(f"{dst}/labels.json", "w"), indent=1)
    size = os.path.getsize(f"{dst}/model.onnx") / 1e9
    print(f"{target}: {len(labels)} labels, int8 model {size:.2f} GB -> {dst}")
    del model
    return labels


for t in TARGETS:
    export(t)

The tokenizer you are loading from '/content/drive/MyDrive/osm-notes-training/ft_v2_context/category/xlm-roberta-large/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
/tmp/ipykernel_4646/2630498726.py:35: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(Logits(model), args, fp32, opset_version=17, dynamo=False, **common)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:196: 

category: 9 labels, int8 model 0.56 GB -> /content/onnx/category


The tokenizer you are loading from '/content/drive/MyDrive/osm-notes-training/ft_v2_context/theme/xlm-roberta-large/model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


theme: 11 labels, int8 model 0.56 GB -> /content/onnx/theme


## 3. Compare int8 against the original model on the test set

`pred` in `predictions.csv` is what the original model said on GPU. If int8
loses less than ~1 point of macro-F1, it is fine to use.

In [4]:
import time

import numpy as np
import onnxruntime as ort
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score


def session(target):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = THREADS
    return ort.InferenceSession(f"{WORK}/{target}/model.onnx", opts, providers=["CPUExecutionProvider"])


def predict(sess, tok, texts, batch=32):
    enc = tok(texts, truncation=True, max_length=MAX_LEN)
    order = np.argsort([len(x) for x in enc["input_ids"]])
    out = [None] * len(texts)
    for s in range(0, len(texts), batch):
        idx = order[s:s + batch]
        b = tok.pad({"input_ids": [enc["input_ids"][i] for i in idx],
                     "attention_mask": [enc["attention_mask"][i] for i in idx]}, return_tensors="np")
        logits = sess.run(["logits"], {"input_ids": b["input_ids"].astype(np.int64),
                                       "attention_mask": b["attention_mask"].astype(np.int64)})[0]
        e = np.exp(logits - logits.max(1, keepdims=True))
        for i, p in zip(idx, e / e.sum(1, keepdims=True)):
            out[i] = p
    return np.stack(out)


rows = []
for t in TARGETS:
    test = pd.read_csv(f"{DRIVE}/{RUN_NAME}/{t}/{MODEL}/predictions.csv", encoding="utf-8-sig",
                       keep_default_na=False)
    test = test[test["split"] == "test"].reset_index(drop=True)
    labels = json.load(open(f"{WORK}/{t}/labels.json"))["labels"]
    tok = AutoTokenizer.from_pretrained(f"{WORK}/{t}")
    sess = session(t)
    t0 = time.time()
    p = predict(sess, tok, test["text"].tolist())
    secs = time.time() - t0
    test["int8"] = np.array(labels)[p.argmax(1)]
    for name, col in (("original (GPU)", "pred"), ("int8 (CPU)", "int8")):
        rows.append({"target": t, "model": name, "n": len(test),
                     "accuracy": round(accuracy_score(test["label"], test[col]), 3),
                     "macro_f1": round(f1_score(test["label"], test[col], average="macro"), 3)})
    rows.append({"target": t, "model": "same answer int8 vs original", "n": len(test),
                 "accuracy": round((test["int8"] == test["pred"]).mean(), 3), "macro_f1": None})
    print(f"{t}: {len(test):,} notes in {secs:.0f} s with {THREADS} threads "
          f"({len(test) / secs:.0f} notes/s)")

summary = pd.DataFrame(rows)
summary

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


category: 2,094 notes in 34 s with 4 threads (61 notes/s)


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


theme: 2,200 notes in 35 s with 4 threads (62 notes/s)


,target,model,n,accuracy,macro_f1
0,category,original (GPU),2094,0.824,0.779
1,category,int8 (CPU),2094,0.826,0.790
2,category,same answer int8 vs original,2094,0.958,NaN
3,theme,original (GPU),2200,0.849,0.825
4,theme,int8 (CPU),2200,0.844,0.820
5,theme,same answer int8 vs original,2200,0.968,NaN


## 4. Save to Drive

Copy the folder to the server (on your computer, Google Drive for desktop syncs it):

```bash
scp -r ".../My Drive/osm-notes-training/ft_v2_context/onnx" \
    root@<server>:/root/osm-notes-deploy/models/classifier/ft_v2_context
```

In [5]:
for t in TARGETS:
    shutil.copytree(f"{WORK}/{t}", f"{OUT}/{t}", dirs_exist_ok=True)
summary.to_csv(f"{OUT}/int8_check.csv", index=False)
!du -sh "{OUT}"/*
print("Saved to", OUT)

558M	/content/drive/MyDrive/osm-notes-training/ft_v2_context/onnx/category
512	/content/drive/MyDrive/osm-notes-training/ft_v2_context/onnx/int8_check.csv
558M	/content/drive/MyDrive/osm-notes-training/ft_v2_context/onnx/theme
Saved to /content/drive/MyDrive/osm-notes-training/ft_v2_context/onnx
